### Declare Libraries

In [ ]:
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Iterator, Sequence
from pathlib import Path
import json
import re
import wave
import numpy as np
import soundfile as sf
import os

import torch
from torch import Tensor, nn
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import BatchSampler
import time

from collections import defaultdict
import matplotlib.pyplot as plt
import random
import shutil
import math
import glob

In [ ]:
def load_mono(path, target_sr=16000):
    # Read the audio
    wav_np, sr = sf.read(path, dtype='float32')
    wav = torch.from_numpy(wav_np).float() 
    
    if wav.ndim == 2:
        wav = wav.mean(dim=1)
        
    # Resample the audio if it is not 16,000 Hz!
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=target_sr)
        
    return wav, target_sr

def stft_magnitude(wav, device, n_fft=512, hop_length=128, win_length=512):
    window = torch.hann_window(win_length, device=device)
    wav = wav.to(device)

    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True,
    )

    # [freq, time] -> [time, freq]
    return spec.transpose(-1, -2)

def hz_to_erb(freq):
    return 21.4 * torch.log10(1.0 + 0.00437 * freq)

def erb_to_hz(erb):
    return (10 ** (erb / 21.4) - 1.0) / 0.00437

def make_erb_filterbank(sample_rate, n_fft, erb_bins, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sample_rate / 2

    erb_edges = torch.linspace(
        hz_to_erb(torch.tensor(low_freq)),
        hz_to_erb(torch.tensor(high_freq)),
        erb_bins + 2,
    )

    hz_edges = erb_to_hz(erb_edges)
    fft_freqs = torch.linspace(0.0, sample_rate / 2, n_fft // 2 + 1)

    filterbank = torch.zeros(erb_bins, n_fft // 2 + 1)

    for i in range(erb_bins):
        left = hz_edges[i]
        center = hz_edges[i + 1]
        right = hz_edges[i + 2]

        rising = (fft_freqs - left) / (center - left)
        falling = (right - fft_freqs) / (right - center)

        filterbank[i] = torch.minimum(rising, falling).clamp_min(0.0)
    return filterbank

## Loss function

In [ ]:
"""Return normalized inverse ERB weights with shape ``[F, E]``."""
def erb_synthesis_matrix(filterbank: Tensor, eps: float = 1e-8) -> Tensor:
    return filterbank.T / filterbank.sum(dim=0, keepdim=True).T.clamp_min(eps)

def apply_erb_gains(input_spec: Tensor, gains: Tensor, synthesis_matrix: Tensor) -> Tensor:
    if gains.ndim != 4 or gains.shape[1] != 1:
        raise ValueError("gains must have shape [B, 1, T, E]")
    """synthesis_matrix: [F, E], which converts values defined on 
    ERB bands back into values for individual FFT frequency bins.
       gains[:, 0]: makes [B, 1, T, E] -> [B, T, E]"""
    frequency_gain = gains[:, 0] @ synthesis_matrix.T    # -> [B, T, F]
    return input_spec * frequency_gain.to(input_spec.dtype)

### This is a derivative of arctan(b, a), where z = a + bi
$$
\delta X = \delta \phi \cdot \left( \frac{-\Im\{X\}}{|X_h|^2}, \frac{\Re\{X\}}{|X_h|^2} \right)
$$

This class is a custom, numerically stable PyTorch function used to calculate the phase (angle) of complex numbers—which is a very common operation when dealing with audio spectrograms (STFT).

Its main purpose is to prevent your model from crashing or outputting NaN (Not a Number) during training when it encounters pure silence.

In [ ]:
class SafeAngle(torch.autograd.Function):
    def forward(ctx, x: Tensor) -> Tensor:
        ctx.save_for_backward(x)
        return torch.atan2(x.imag, x.real)

    def backward(ctx, grad: Tensor) -> Tensor:
        (x,) = ctx.saved_tensors
        # |X_h|^2 = max(Re^2 + Im^2, 1e-12)
        denom = (x.real.square() + x.imag.square()).clamp_min(1e-12)
        grad_scaled = grad / denom
        return torch.view_as_complex(torch.stack([-x.imag * grad_scaled, x.real * grad_scaled], dim=-1))

safe_angle = SafeAngle.apply

### 1. Compressed Spectral Loss and ERB Mask Loss

In [ ]:
class EnhancedSpectralLoss(nn.Module):
    def __init__(self, power: float = 0.6, factor_magnitude: float = 1.0, factor_complex: float = 1.0, factor_under: float = 2.0):
        super().__init__()
        self.power = power
        self.factor_magnitude = factor_magnitude
        self.factor_complex = factor_complex
        self.factor_under = factor_under

    def forward(self, predicted_spec: Tensor, target_spec: Tensor):
        pred_mag = predicted_spec.abs().clamp_min(1e-8).pow(self.power)
        target_mag = target_spec.abs().clamp_min(1e-8).pow(self.power)

        mag_err = (pred_mag - target_mag).square()
        if self.factor_under != 1.0:
            # Overpenalize when prediction < target (speech was cut)
            under_weight = torch.where(pred_mag < target_mag, self.factor_under, 1)
            mag_err = mag_err * under_weight

        loss_mag = mag_err.mean() * self.factor_magnitude

        loss_complex = torch.tensor(0.0, device=predicted_spec.device)
        if self.factor_complex > 0:
            pred_c = pred_mag * torch.exp(1j * safe_angle(predicted_spec))
            target_c = target_mag * torch.exp(1j * safe_angle(target_spec))
            loss_complex = F.mse_loss(torch.view_as_real(pred_c), torch.view_as_real(target_c)) * self.factor_complex

        return loss_mag + loss_complex

class ERBMaskLoss(nn.Module):
    """
    Direct supervision on the ERB mask using Ideal Amplitude Mask (IAM)
    Compute the loss of output gains with ideal gains/IAM
    Uses L2 + L4 penalties to heavily penalize large gain errors
    """
    def __init__(self, power_compress: float = 0.6, factor_under: float = 2.0, factor_l2: float = 1.0, factor_l4: float = 10.0, eps: float = 1e-12):
        super().__init__()
        self.power_compress = power_compress
        self.factor_under = factor_under
        self.factor_l2 = factor_l2
        self.factor_l4 = factor_l4
        self.eps = eps

    def forward(self, predicted_gains: Tensor, input_spec: Tensor, target_spec: Tensor, filterbank: Tensor) -> Tensor:
        input_pow = input_spec.abs().square()
        target_pow = target_spec.abs().square()

        fb = filterbank.to(input_spec.device).T
        input_erb_pow = input_pow @ fb
        target_erb_pow = target_pow @ fb

        # Ideal Amplitude Mask in ERB domain: sqrt(S_power / (X_power + eps))
        target_mask = (target_erb_pow / (input_erb_pow + self.eps)).sqrt().clamp(0.0, 1.0)
        target_mask = target_mask.unsqueeze(1) # [B, 1, T, E]

        # Power compress target and prediction for dynamic range equalization
        g_pred = predicted_gains.clamp_min(self.eps).pow(self.power_compress)
        g_true = target_mask.clamp_min(self.eps).pow(self.power_compress)

        diff = g_pred - g_true
        if self.factor_under != 1.0:
            diff = diff * torch.where(g_pred < g_true, self.factor_under, 1.0)

        # L2 and L4 norm penalty
        loss_l2 = diff.square().clamp_min(1e-13).mean() * self.factor_l2
        loss_l4 = diff.square().clamp_min(1e-13).pow(2).mean() * self.factor_l4

        return loss_l2 + loss_l4

### 2. Combined Loss

In [ ]:
class DeepFilterNetCombinedLoss(nn.Module):
    def __init__(
        self,
        filterbank: Tensor,
        lambda_mask: float = 0.5,
        lambda_stage1: float = 0.5,
    ):
        super().__init__()
        self.spectral_loss = EnhancedSpectralLoss(power=0.3, factor_under=2.0)
        self.mask_loss = ERBMaskLoss(power_compress=0.3, factor_under=2.0)

        self.lambda_stage1 = lambda_stage1
        self.lambda_mask = lambda_mask
        self.register_buffer("filterbank", filterbank)

    def forward(self,
                stage1_spec: Tensor, # [B, T, F] (Y^G)
                target_spec: Tensor, # [B, T, F] Clean target (S)
                input_spec: Tensor, # [B, T, F] Noisy input (X)
                pred_gains: Tensor, # [B, 1, T, erb_bins]
               ) -> Tensor:

        l_stage1 = self.spectral_loss(stage1_spec, target_spec)
        
        l_mask = self.mask_loss(pred_gains, input_spec, target_spec, self.filterbank)
        
        
        total_loss = self.lambda_mask * l_mask + self.lambda_stage1 * l_stage1

        return total_loss

### Set up layers

In [ ]:
def channel_shuffle(x: Tensor, groups: int) -> Tensor:
    b, t, d = x.shape
    if d % groups != 0:
        raise ValueError("Feature dimension must be divisible by groups")

    features_per_group = d // groups
    x = x.view(b, t, groups, features_per_group)
    x = x.transpose(2, 3).contiguous()
    x = x.view(b, t, d)

    return x

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, kernel_size: Tuple[int, int] = (3, 2), stride: Tuple[int, int] = (1, 1), lookahead: int = 0) -> None:
        super().__init__()

        kt, kf = kernel_size

        if lookahead < 0 or lookahead > kt - 1:
            raise ValueError("lookahead must satisfy 0 <= lookahead <= kt - 1")

        time_left = kt - 1 - lookahead
        time_right = lookahead
        self.pad = (
            kf // 2, # left
            kf - 1 - kf // 2, # right
            time_left, # top
            time_right, # bottom
        )

        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
            groups=in_channels,
            bias=False,
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU()
    def forward(self, x: Tensor) -> Tensor:
        x = F.pad(x, self.pad)
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        x = self.act(x)
        return x
    
class SeparableTConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, scale_factor: int = 2, lookahead: int = 0) -> None:
        super().__init__()
        self.scale_factor = scale_factor

        self.conv = SeparableConv2d(in_channels, out_channels, kernel_size=(3, 2), lookahead=lookahead)

    def forward(self, x: Tensor) -> Tensor:
        x = F.interpolate(
            x, 
            scale_factor=(1, self.scale_factor), # [width, height] ~ [time, frequency]
            mode="nearest"
        )
        return self.conv(x)

class GroupedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()

        if in_features % groups:
            raise ValueError("in_features must be divisible by groups")

        if out_features % groups:
            raise ValueError("out_features must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.in_per_group = in_features // groups
        self.out_per_group = out_features // groups

        self.layers = nn.ModuleList(
            [
                nn.Linear(
                    self.in_per_group,
                    self.out_per_group,
                )
                for _ in range(groups)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, T, D]

        chunks = x.split(self.in_per_group, dim=-1)

        output_chunks = []

        for layer, chunk in zip(self.layers, chunks):
            y = layer(chunk)
            output_chunks.append(y)

        x = torch.cat(output_chunks, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRU(nn.Module):
    def __init__(self, input_size: int = 512, hidden_size: int = 512, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()
        if input_size % groups:
            raise ValueError("input_size must be divisible by groups")
        if hidden_size % groups:
            raise ValueError("hidden_size must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.input_per_group = input_size // groups
        self.hidden_per_groups = hidden_size // groups

        self.grus = nn.ModuleList(
            [
                nn.GRU(
                    input_size=self.input_per_group,
                    hidden_size=self.hidden_per_groups,
                    batch_first=True,
                )
                for _ in range(groups)
            ]
        )


    def forward(self, x: Tensor) -> Tensor:
        # [B, T, D]
        chunks = x.split(self.input_per_group, dim=-1)
        outputs = []

        for gru, chunk in zip(self.grus, chunks):
            # GRU returns [output, hidden]
            y, _ = gru(chunk)
            outputs.append(y)

        x = torch.cat(outputs, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRUStack(nn.Module):
    def __init__(self, size: int = 512, groups: int = 8, num_layers: int = 3) -> None:
        super().__init__()

        self.layers = nn.ModuleList(
            [
                GroupedGRU(
                    input_size=size,
                    hidden_size=size,
                    groups=groups,
                    shuffle=True,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers:
            x = layer(x)

        return x

class PConv(nn.Module):
    def __init__(self, channels: int = 64, out_channels: int = 64) -> None:
        super().__init__()

        self.conv = nn.Conv2d(channels, out_channels, kernel_size=1, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x)

## Implement ERB encoder, decoder

In [ ]:
class ERBEncoder(nn.Module):
    def __init__(self, erb_bins: int = 32, channels: int = 64, hidden_size: int = 512, groups: int = 8, conv_lookahead: int = 2) -> None:
        super().__init__()

        if erb_bins % 8:
            raise ValueError("erb_bins must be divisible by 8")

        self.erb_bins = erb_bins
        self.channels = channels
        self.conv_lookahead = conv_lookahead
        lookahead0 = 1 if conv_lookahead > 0 else 0
        lookahead1 = 1 if conv_lookahead > 1 else 0
        lookahead2 = 1 if conv_lookahead > 2 else 0
        self.conv0 = SeparableConv2d(1, channels, lookahead=lookahead0)
        self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=lookahead1) # B -> B / 2
        self.conv2 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=lookahead2)
        self.conv3 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=0)
        self.glinear = GroupedLinear(in_features=channels*erb_bins//8, out_features=hidden_size, groups=groups)
        self.gru = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=3)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
        x0 = self.conv0(x)
        x1 = self.conv1(x0)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)

        batch, channels, time, freq = x3.shape
        x = x3.permute(0, 2, 1, 3)
        x = x.reshape(batch, time, channels*freq)

        x = self.glinear(x)
        embedding = self.gru(x)
        embedding = self.dropout(embedding)

        return x0, x1, x2, x3, embedding

class ERBDecoder(nn.Module):
    def __init__(self, channels: int = 64, erb_bins: int = 32, hidden_size: int = 512) -> None:
        super().__init__()

        bottleneck_freq = erb_bins // 8
        bottleneck_size = channels * bottleneck_freq

        self.channels = channels
        self.bottelneck_freq = bottleneck_freq

        self.linear = GroupedLinear(hidden_size, bottleneck_size)

        self.p3 = PConv(channels)
        self.p2 = PConv(channels)
        self.p1 = PConv(channels)
        self.p0 = PConv(channels)

        self.up3 = SeparableTConv2d(channels, channels, lookahead=0)
        self.up2 = SeparableTConv2d(channels, channels, lookahead=0)
        self.up1 = SeparableTConv2d(channels, channels, lookahead=0)

        self.final_conv = SeparableConv2d(channels, 1, kernel_size=(3, 2), lookahead=0)
        self.output_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, embedding: Tensor, x0: Tensor, x1: Tensor, x2: Tensor, x3: Tensor) -> Tensor:
        batch, time, _ = embedding.shape

        x = self.linear(embedding)
        x = self.dropout(x)

        x = x.reshape(batch, time, self.channels, self.bottelneck_freq)
        x = x.permute(0, 2, 1, 3).contiguous()

        p3_x3 = self.p3(x3)
        x = x + p3_x3

        x = self.up3(x)
        x = x + self.p2(x2)

        x = self.up2(x)
        x = x + self.p1(x1)

        x = self.up1(x)
        x = x + self.p0(x0)

        gains = self.final_conv(x)
        gains = self.output_activation(gains)

        return gains

## Deep Neural Network (DNN)

In [ ]:
class DeepFilterNetDNN(nn.Module):
    def __init__(self, erb_bins: int = 32, channels: int = 64, hidden_size: int = 512, groups: int = 8, conv_lookahead: int = 2) -> None:
        super().__init__()
        
        self.encoder = ERBEncoder(
            erb_bins=erb_bins,
            channels=channels,
            hidden_size=hidden_size,
            groups=groups,
            conv_lookahead=conv_lookahead,
        )

        self.decoder = ERBDecoder(
            channels=channels,
            erb_bins=erb_bins,
            hidden_size=hidden_size
        )

    def forward(self, erb_features: Tensor) -> Tensor:
        # Stage 1:
        e0, e1, e2, e3, embedding = self.encoder(erb_features)
        gains = self.decoder(embedding, e0, e1, e2, e3)

        return gains


## On-the-fly augmentation - Dataset Generator
Instead of pre-mixing and saving thousands of noisy audio files to the hard drive (which would take up terabytes of storage), this section defines a dynamic dataset generator. During training, as the neural network requests a new batch of data, the generator dynamically creates unique audio samples "on-the-fly" by:

1. Randomly selecting a clean speech file.
2. Applying a random Room Impulse Response (RIR) to simulate physical environments.
3. Mixing in a random noise file at varying Signal-to-Noise Ratios (SNR).

This approach saves massive amounts of disk space and ensures the AI model rarely sees the exact same acoustic combination twice, acting as powerful data augmentation to prevent overfitting.

In [ ]:
class DynamicMixingDataset(Dataset):
    def __init__(self, clean_dir, noise_dir, rir_dir, filterbank, segment_samples=48000, flag: bool = False):
        self.clean_files = sorted(glob.glob(f"{clean_dir}/**/*.wav", recursive=True))
        self.noise_files = sorted(glob.glob(f"{noise_dir}/**/*.wav", recursive=True))
        self.rir_files = sorted(glob.glob(f"{rir_dir}/**/*.wav", recursive=True))
        if rir_dir is not None:
            self.rir_files = sorted(glob.glob(f"{rir_dir}/**/*.wav", recursive=True))
        else:
            self.rir_files = []

        self.segment_samples = segment_samples
        self.filterbank = filterbank

        self.snr_levels = [-5.0, 0.0, 5.0, 10.0, 20.0, 40.0]
        self.random_gains = [-6.0, 0.0, 6.0]
        self.flag = flag

    def __len__(self):
        return len(self.clean_files)

    def _load_and_crop(self, path, rng, target_sr=16000, target_samples=48000):
        # Read directly from the path!
        wav_np, sr = sf.read(path, dtype='float32')
        wav = torch.from_numpy(wav_np).float()
        
        if wav.ndim == 2:
            wav = wav.mean(dim=1)
            
        orig_length = int(target_samples * (sr / target_sr))
        
        # Crop first
        if wav.shape[-1] > orig_length:
            start = rng.randint(0, wav.shape[-1] - orig_length)
            wav = wav[start : start + orig_length]
        else:
            wav = torch.nn.functional.pad(wav, (0, orig_length - wav.shape[-1]))
            
        # Resample second
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=target_sr)
            
        return wav.unsqueeze(0)
        
    def __getitem__(self, idx):
        if self.flag:
            rng = random.Random(42 + idx)
        else:
            rng = random.Random()
        
        clean_wav = self._load_and_crop(self.clean_files[idx], rng, target_samples=self.segment_samples)
        speech_gain_db = rng.choice(self.random_gains)
        speech_gain_linear = 10 ** (speech_gain_db / 20.0)
        clean_wav = clean_wav * speech_gain_linear

        # Apply RIR (Reverberation) 50% of the time
        if len(self.rir_files) > 0 and rng.random() < 0.5:
            rir_idx = rng.randint(0, len(self.rir_files) - 1)
            rir_wav = self._load_and_crop(self.rir_files[rir_idx], rng, target_samples=16000)

            # Use max 1 second of reverb tail to keep math fast
            if rir_wav.shape[-1] > 16000:
                rir_wav = rir_wav[:, :16000]

            clean_reverb = torchaudio.functional.fftconvolve(clean_wav, rir_wav)
            clean_reverb = clean_reverb[:, :self.segment_samples]

            # Normalize to match original clean energy level so it doesn't artificially clip
            clean_reverb = clean_reverb / (torch.max(torch.abs(clean_reverb)) + 1e-8) * torch.max(torch.abs(clean_wav))
        else:
            clean_reverb = clean_wav


        """Create complex noise mixture (up to 5 noise signals)"""
        noise_mixture = torch.zeros_like(clean_wav)
        num_noises = rng.randint(1, 5)

        for _ in range(num_noises):
            n_idx = rng.randint(0, len(self.noise_files) - 1)
            n_wav = self._load_and_crop(self.noise_files[n_idx], rng, target_samples=self.segment_samples)

            gain_db = rng.choice(self.random_gains)
            gain_linear = 10 ** (gain_db / 20.0)

            noise_mixture += (n_wav * gain_linear)

        """Mix clean speech and mixture of noise at chosen SNR level"""
        target_snr_db = rng.choice(self.snr_levels)

        clean_power = torch.mean(clean_reverb ** 2)
        noise_power = torch.mean(noise_mixture ** 2)

        if noise_power > 0:
            snr_linear = 10 ** (target_snr_db / 10.0)
            final_noise_scalar = math.sqrt(clean_power / (noise_power * snr_linear + 1e-8))
            noise_mixture = noise_mixture * final_noise_scalar

        mixed_wav = clean_reverb + noise_mixture

        # Prevent hard clipping bounds
        max_val = torch.max(torch.abs(mixed_wav))
        if max_val > 0.99:
            mixed_wav = (mixed_wav / max_val) * 0.95
            clean_reverb = (clean_reverb / max_val) * 0.95

        window = torch.hann_window(512)

        input_spec = torch.stft(mixed_wav.squeeze(), n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
        target_spec = torch.stft(clean_reverb.squeeze(), n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)

        input_erb = torch.matmul(input_spec.abs().square().transpose(0, 1), self.filterbank.T)
        input_erb = torch.log10(input_erb + 1e-7)

        return {
            "input_spec": input_spec.transpose(0, 1),  # [T, F]
            "target_spec": target_spec.transpose(0, 1),
            "input_erb": input_erb.unsqueeze(0),    #[1, T, F]
        }

## Define Parameters

In [ ]:
sample_rate = 16000
n_fft = 512
hop_length = 128
win_length = 512
erb_bins = 32
batch_size = 20
epochs = 100
learning_rate = 1e-3
df_bins = 96
df_order = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
filterbank = make_erb_filterbank(sample_rate=sample_rate, n_fft=n_fft, erb_bins=erb_bins)
criterion = DeepFilterNetCombinedLoss(filterbank=filterbank).to(device)


## Dataloader

In [ ]:
train_dataset = DynamicMixingDataset(
    clean_dir="/kaggle/input/datasets/quanninhhoang/dns-cleanspeech-dataset/train",
    noise_dir="/kaggle/input/datasets/quanninhhoang/dns-noise-dataset/train",
    rir_dir="/kaggle/input/datasets/quanninhhoang/dns-impulse-dataset/train",
    filterbank=filterbank
)

val_dataset = DynamicMixingDataset(
    clean_dir="/kaggle/input/datasets/quanninhhoang/dns-cleanspeech-dataset/val",
    noise_dir="/kaggle/input/datasets/quanninhhoang/dns-noise-dataset/val",
    rir_dir="/kaggle/input/datasets/quanninhhoang/dns-impulse-dataset/val",
    filterbank=filterbank,
    flag=True
)

test_dataset = DynamicMixingDataset(
    clean_dir="/kaggle/input/datasets/quanninhhoang/dns-cleanspeech-dataset/test",
    noise_dir="/kaggle/input/datasets/quanninhhoang/dns-noise-dataset/test",
    rir_dir=None,
    filterbank=filterbank,
    segment_samples=160000,
    flag=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)


## Validation Function

In [ ]:
def evaluate_model(model, loader, synthesis_matrix, device):
    model.eval()
    total_val_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch in loader:
            input_erb = batch["input_erb"].to(device, dtype=torch.float32, non_blocking=True)
            input_spec = batch["input_spec"].to(device, non_blocking=True)
            target_spec = batch["target_spec"].to(device, non_blocking=True)

            mean = input_erb.mean(dim=-2, keepdim=True)
            std = input_erb.std(dim=-2, keepdim=True).clamp_min(1e-4)
            input_erb = (input_erb - mean) / std

            # 1. Model parameter estimation
            gains = model(input_erb)

            # 2. Stage 1: ERB envelope filtering
            stage1_spec = apply_erb_gains(input_spec, gains, synthesis_matrix)

            # 4. Loss computation
            loss = criterion(
                stage1_spec=stage1_spec,
                target_spec=target_spec,
                input_spec=input_spec,
                pred_gains=gains,
            )
            
            total_val_loss += loss.item()
            total_batches += 1

    return total_val_loss / max(total_batches, 1)


## Training Step

In [ ]:
def train_model(model, train_loader, val_loader, synthesis_matrix, optimizer, scheduler, epochs, checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_val_loss = float("inf")
    history = []

    for epoch in range(epochs):
        epoch_start_time = time.perf_counter()
        model.train()

        total_train_loss = 0.0
        total_batches = 0

        for batch in train_loader:
            input_erb = batch["input_erb"].to(device, dtype=torch.float32, non_blocking=True)
            input_spec = batch["input_spec"].to(device, non_blocking=True)
            target_spec = batch["target_spec"].to(device, non_blocking=True)

            mean = input_erb.mean(dim=-2, keepdim=True)
            std = input_erb.std(dim=-2, keepdim=True).clamp_min(1e-4)
            input_erb = (input_erb - mean) / std
            
            optimizer.zero_grad(set_to_none=True)

            # 1. Model parameter estimation
            gains = model(input_erb)

            # 2. Stage 1: ERB envelope filtering
            stage1_spec = apply_erb_gains(input_spec, gains, synthesis_matrix)

            # 3. Loss computation
            loss = criterion(
                stage1_spec=stage1_spec,
                target_spec=target_spec,
                input_spec=input_spec,
                pred_gains=gains,
            )
            
            loss.backward()
            optimizer.step()
            total_train_loss += loss.detach().item()
            total_batches += 1
            
        train_loss = (total_train_loss / max(total_batches, 1))
        
        val_loss = evaluate_model(
            model=model,
            loader=val_loader,
            synthesis_matrix=synthesis_matrix,
            device=device,
        )

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        epoch_duration = time.perf_counter() - epoch_start_time
        mins, secs = divmod(int(epoch_duration), 60)
        time_str = f"{mins:02d}m {secs:02d}s"
        
        history.append(
            {
                "epoch": epoch+1,
                "train_loss": train_loss,
                "val_loss": val_loss,
            }
        )
        epoch_checkpoint = checkpoint_dir / (f"epoch_{epoch + 1:03d}.ckpt")
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
            },
            epoch_checkpoint,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_checkpoint = checkpoint_dir / (f"best_model.ckpt")
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                },
                best_checkpoint,
            )
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} [{time_str}] "
                f"train={train_loss:.6f} "
                f"val={val_loss:.6f} "
                f"best checkpoint saved",
                flush=True,
            )
        else:
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} [{time_str}] "
                f"train={train_loss:.6f} "
                f"val={val_loss:.6f}",
                flush=True,
            )

    return history


In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

model = DeepFilterNetDNN(erb_bins=erb_bins).to(device)
synthesis_matrix = erb_synthesis_matrix(filterbank).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [ ]:
checkpoint_dir = "/kaggle/working/erb_checkpoints"

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    synthesis_matrix=synthesis_matrix,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
    checkpoint_dir=checkpoint_dir,
)

## Visualize Training and Validation Loss

In [ ]:
epochs_plot = [
    item["epoch"]
    for item in history
]

train_loss = [
    item["train_loss"]
    for item in history
]

val_loss = [
    item["val_loss"]
    for item in history
]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_plot,
    train_loss,
    color="red",
    marker="o",
    label="Training Loss"
)
plt.plot(
    epochs_plot,
    val_loss,
    color="blue",
    marker="s",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()